In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
path = r"C:\Data Analytics\Projects\END TO END\Marketing Campaigns Analysis\Dataset"

In [3]:
df_campaigns = pd.read_csv(path+"\campaigns.csv")
df_performance = pd.read_csv(path +"\campaign_performance.csv")
df_influencer = pd.read_csv(path + "\influencer_affiliate.csv")

In [4]:
print(f"df_campaigns          : {df_campaigns.shape[0]:,} rows × {df_campaigns.shape[1]} columns")
print(f"df_performance: {df_performance.shape[0]:,} rows × {df_performance.shape[1]} columns")
print(f"df_influencer: {df_influencer.shape[0]:,} rows × {df_influencer.shape[1]} columns")

df_campaigns          : 800 rows × 12 columns
df_performance: 242,487 rows × 15 columns
df_influencer: 25,000 rows × 13 columns


# Initial Data Inspection 

In [6]:
for name, df in [('CAMPAIGNS', df_campaigns), 
                ('PERFORMACE', df_performance), 
                ('INFLUENCER', df_influencer)]: 
    print(f"\n{'='*50}")
    print(f"  TABLE: {name}")
    print(f"{'='*50}")
    print(f"\n📌 Data Types & Missing Values:\n")

    inspection = pd.DataFrame({
        'dtype':df.dtypes,
        'Missing_Count': df.isnull().sum(), 
        'Missing_Pct' : (df.isnull().sum()/len(df)*100).round(2)
    })

    print(inspection)
    print(f"\n📌 Sample (Top 2 rows):\n")
    print(df.head(2))


  TABLE: CAMPAIGNS

📌 Data Types & Missing Values:

                   dtype  Missing_Count  Missing_Pct
Campaign_ID       object              0          0.0
Campaign_Name     object              0          0.0
Channel           object              0          0.0
Campaign_Type     object              0          0.0
Target_Audience   object              0          0.0
Region            object              0          0.0
Start_Date        object              0          0.0
End_Date          object              0          0.0
Total_Budget       int64              0          0.0
Product_Category  object              0          0.0
Client_Name       object              0          0.0
Campaign_Status   object              0          0.0

📌 Sample (Top 2 rows):

  Campaign_ID                            Campaign_Name          Channel  \
0    CMP_0001   Home & Living_Conversion_Campaign_0001              SEO   
1    CMP_0002  Education_Lead Generation_Campaign_0002  Email Marketing   

     Ca

### Step 2 Complete: Initial Inspection
- Zero missing values across all 3 tables ✅
- Numeric columns (int64, float64) — correct dtypes ✅
- Date columns (Start_Date, End_Date, Date, Post_Date) — currently object, need datetime64 fix ⚠️

In [7]:
camp_cols = df_campaigns.columns.tolist()
inf_col = df_influencer.columns.tolist()
per_col = df_performance.columns.tolist()

# Fix Date Columns — object → datetime64

In [10]:
inf_col

['Record_ID',
 'Campaign_ID',
 'Partner_Type',
 'Partner_Name',
 'Platform',
 'Influencer_Tier',
 'Reach',
 'Engagement_Rate',
 'Clicks',
 'Conversions',
 'Commission_Paid',
 'Revenue_Generated',
 'Post_Date']

In [11]:
df_campaigns['Start_Date'] = pd.to_datetime(df_campaigns['Start_Date'])
df_campaigns['End_Date'] = pd.to_datetime(df_campaigns['End_Date'])

df_performance['Date'] = pd.to_datetime(df_performance['Date'])

df_influencer['Post_Date'] = pd.to_datetime(df_influencer['Post_Date'])

print("✅ Date columns fixed!")
print("=" * 40)
print(f"Start_Date dtype : {df_campaigns['Start_Date'].dtype}")
print(f"End_Date dtype   : {df_campaigns['End_Date'].dtype}")
print(f"Date dtype       : {df_performance['Date'].dtype}")
print(f"Post_Date dtype  : {df_influencer['Post_Date'].dtype}")

✅ Date columns fixed!
Start_Date dtype : datetime64[ns]
End_Date dtype   : datetime64[ns]
Date dtype       : datetime64[ns]
Post_Date dtype  : datetime64[ns]


# Check & Remove Duplicates

In [12]:
print(f"campaigns    : {df_campaigns.duplicated().sum()} duplicates")
print(f"performance  : {df_performance.duplicated().sum()} duplicates")
print(f"influencer   : {df_influencer.duplicated().sum()} duplicates")

campaigns    : 0 duplicates
performance  : 0 duplicates
influencer   : 0 duplicates


 # Feature Engineering: Campaigns Table

## Feature 1: Campaign Duration in Days

In [13]:
df_campaigns['Campaign_Duration_Days'] = (
    df_campaigns['End_Date'] - df_campaigns['Start_Date']
).dt.days 

## Feature 2: Budget Category 

In [14]:
df_campaigns['Budget_Category'] = pd.cut(
    df_campaigns['Total_Budget'], 
    bins= [0,200000,800000, float('inf')], 
    labels= ['Small','Medium', 'Large']
)

In [15]:
print(f"Campaign_Duration_Days — Min : {df_campaigns['Campaign_Duration_Days'].min()} days")
print(f"Campaign_Duration_Days — Max : {df_campaigns['Campaign_Duration_Days'].max()} days")
print(f"Campaign_Duration_Days — Mean: {df_campaigns['Campaign_Duration_Days'].mean():.1f} days")

Campaign_Duration_Days — Min : 200 days
Campaign_Duration_Days — Max : 400 days
Campaign_Duration_Days — Mean: 302.1 days


In [16]:
print(f"\nBudget_Category Distribution:")
print(df_campaigns['Budget_Category'].value_counts())
print(f"\nSample (Top 3 rows — new columns):")
print(df_campaigns[['Campaign_ID', 'Total_Budget', 'Budget_Category', 
                     'Start_Date', 'End_Date', 'Campaign_Duration_Days']].head(3))


Budget_Category Distribution:
Budget_Category
Small     392
Medium    282
Large     126
Name: count, dtype: int64

Sample (Top 3 rows — new columns):
  Campaign_ID  Total_Budget Budget_Category Start_Date   End_Date  \
0    CMP_0001        331932          Medium 2022-11-24 2023-09-30   
1    CMP_0002        171958           Small 2022-02-27 2022-12-16   
2    CMP_0003        171958           Small 2022-01-13 2023-01-10   

   Campaign_Duration_Days  
0                     310  
1                     292  
2                     362  


### Feature Engineering — Campaigns Table
- Campaign_Duration_Days → Min: 200, Max: 400, Mean: 302.1 days
- Budget_Category → Small: 392 (49%), Medium: 282 (35%), Large: 126 (16%)
- 2 new columns added successfully ✅

# Feature Engineering: Performance Table

## Feature 1: ROI% 

In [18]:
per_col

['Performance_ID',
 'Campaign_ID',
 'Date',
 'Impressions',
 'Clicks',
 'Conversions',
 'Spend',
 'Revenue',
 'CTR',
 'CVR',
 'CPC',
 'CPA',
 'ROAS',
 'Bounce_Rate',
 'Avg_Session_Duration']

In [19]:
df_performance['ROI_Pct'] = df_performance.apply(
    lambda row : round((row['Revenue'] - row['Spend'])/row['Spend']*100,2),
    axis = 1 
)

## Feature 2: ROAS Category

In [20]:
df_performance['ROAS_Category'] = pd.cut(
    df_performance['ROAS'],
    bins   = [0, 2, 4, 7, float('inf')],
    labels = ['Poor', 'Average', 'Good', 'Excellent']
)

## Feature 3: CTR Category

In [21]:
df_performance['CTR_Category'] = pd.cut(
    df_performance['CTR'],
    bins   = [0, 2, 5, float('inf')],
    labels = ['Low', 'Medium', 'High']
)

## Feature 4: Time columns from Date

In [22]:
df_performance['Month']   = df_performance['Date'].dt.month
df_performance['Quarter'] = df_performance['Date'].dt.quarter
df_performance['Year']    = df_performance['Date'].dt.year

## Feature 5: Revenue Per Impression

In [23]:
df_performance['Revenue_Per_Impression'] = (
    df_performance['Revenue'] / df_performance['Impressions']
).round(4)

In [24]:
print("✅ Feature Engineering — Performance Table Complete!")
print("=" * 55)
print(f"New columns added: ROI_Pct, ROAS_Category, CTR_Category,")
print(f"                   Month, Quarter, Year, Revenue_Per_Impression")
print(f"\nROI_Pct        — Min : {df_performance['ROI_Pct'].min():.2f}%")
print(f"ROI_Pct        — Max : {df_performance['ROI_Pct'].max():.2f}%")
print(f"ROI_Pct        — Mean: {df_performance['ROI_Pct'].mean():.2f}%")
print(f"\nROAS_Category Distribution:")
print(df_performance['ROAS_Category'].value_counts().sort_index())
print(f"\nCTR_Category Distribution:")
print(df_performance['CTR_Category'].value_counts().sort_index())
print(f"\nTotal columns now: {df_performance.shape[1]}")

✅ Feature Engineering — Performance Table Complete!
New columns added: ROI_Pct, ROAS_Category, CTR_Category,
                   Month, Quarter, Year, Revenue_Per_Impression

ROI_Pct        — Min : 50.02%
ROI_Pct        — Max : 1399.98%
ROI_Pct        — Mean: 482.48%

ROAS_Category Distribution:
ROAS_Category
Poor          5154
Average      76895
Good         98579
Excellent    61859
Name: count, dtype: int64

CTR_Category Distribution:
CTR_Category
Low        11023
Medium     96740
High      134724
Name: count, dtype: int64

Total columns now: 22


### Feature Engineering — Performance Table
- ROI_Pct → Min: 50.02%, Max: 1399.98%, Mean: 482.48%
- ROAS_Category → Poor: 5154 | Average: 76895 | Good: 98579 | Excellent: 61859
- CTR_Category → Low: 11023 | Medium: 96740 | High: 134724
- Month, Quarter, Year extracted from Date ✅
- Revenue_Per_Impression added ✅
- Total columns: 15 → 22 ✅

In [30]:
df_influencer_agg = df_influencer.groupby('Campaign_ID').agg(
    Total_Reach          = ('Reach',              'sum'),
    Avg_Engagement_Rate  = ('Engagement_Rate',    'mean'),
    Total_Clicks_inf     = ('Clicks',             'sum'),
    Total_Conversions_inf= ('Conversions',        'sum'),
    Total_Commission     = ('Commission_Paid',    'sum'),
    Total_Revenue_inf    = ('Revenue_Generated',  'sum'),
    Partner_Count        = ('Record_ID',          'count')
).reset_index()


print(f"Shape: {df_influencer_agg.shape[0]:,} rows × {df_influencer_agg.shape[1]} columns")
print(f"\nSample:")
print(df_influencer_agg.head(3))

Shape: 800 rows × 8 columns

Sample:
  Campaign_ID  Total_Reach  Avg_Engagement_Rate  Total_Clicks_inf  \
0    CMP_0001     68135294             4.756071            602053   
1    CMP_0002     76674234             5.707273            714298   
2    CMP_0003     75659861             4.429706            895206   

   Total_Conversions_inf  Total_Commission  Total_Revenue_inf  Partner_Count  
0                  67493         650097.68         2901471.38             28  
1                  68334         923733.02         3368172.22             33  
2                 100875         818458.19         3721231.88             34  


# Merge Tables

In [31]:
df_master = pd.merge(
    df_performance,
    df_campaigns,
    on  = 'Campaign_ID',
    how = 'left'
)

print("✅ Step 1: Performance + Campaigns merged!")
print(f"Shape: {df_master.shape[0]:,} rows × {df_master.shape[1]} columns") 

✅ Step 1: Performance + Campaigns merged!
Shape: 242,487 rows × 35 columns


In [32]:
df_master = pd.merge(
    df_master,
    df_influencer_agg,
    on  = 'Campaign_ID',
    how = 'left'
)

print("\n✅ Step 2: Master + Influencer (aggregated) merged!")
print(f"Shape: {df_master.shape[0]:,} rows × {df_master.shape[1]} columns")
print(f"\n📌 Final marketing_master:")
print(f"Rows    : {df_master.shape[0]:,}")
print(f"Columns : {df_master.shape[1]}")
print(f"\nAll Columns:")
print(df_master.columns.tolist())


✅ Step 2: Master + Influencer (aggregated) merged!
Shape: 242,487 rows × 42 columns

📌 Final marketing_master:
Rows    : 242,487
Columns : 42

All Columns:
['Performance_ID', 'Campaign_ID', 'Date', 'Impressions', 'Clicks', 'Conversions', 'Spend', 'Revenue', 'CTR', 'CVR', 'CPC', 'CPA', 'ROAS', 'Bounce_Rate', 'Avg_Session_Duration', 'ROI_Pct', 'ROAS_Category', 'CTR_Category', 'Month', 'Quarter', 'Year', 'Revenue_Per_Impression', 'Campaign_Name', 'Channel', 'Campaign_Type', 'Target_Audience', 'Region', 'Start_Date', 'End_Date', 'Total_Budget', 'Product_Category', 'Client_Name', 'Campaign_Status', 'Campaign_Duration_Days', 'Budget_Category', 'Total_Reach', 'Avg_Engagement_Rate', 'Total_Clicks_inf', 'Total_Conversions_inf', 'Total_Commission', 'Total_Revenue_inf', 'Partner_Count']


### Step 7 Complete: 3 Tables Merged → marketing_master
- Step 1: Performance + Campaigns → 2,42,487 rows × 33 columns
- Step 2: Master + Influencer (aggregated) → 2,42,487 rows × 42 columns
- Influencer table pehle Campaign_ID level pe aggregate kiya (many-to-many problem solve)
- Final marketing_master: 2,42,487 rows × 42 columns ✅

# Saving Files 

In [33]:
df_campaigns.to_csv(r"C:\Data Analytics\Projects\END TO END\Marketing Campaigns Analysis\Dataset\Cleaned Dataset\Campaigns_cleaned.csv", index=False)
df_performance.to_csv('C:\Data Analytics\Projects\END TO END\Marketing Campaigns Analysis\Dataset\Cleaned Dataset\performance_cleaned.csv', index=False)
df_influencer.to_csv('C:\Data Analytics\Projects\END TO END\Marketing Campaigns Analysis\Dataset\Cleaned Dataset\influencer_cleaned.csv',   index=False)

# Save master table
df_master.to_csv('C:\Data Analytics\Projects\END TO END\Marketing Campaigns Analysis\Dataset\Cleaned Dataset\marketing_master.csv', index=False)


In [34]:
print("✅ All files saved successfully!")
print("=" * 40)
print("📁 Files created:")
print("   1. campaigns_cleaned.csv")
print("   2. performance_cleaned.csv")
print("   3. influencer_cleaned.csv")
print("   4. marketing_master.csv  ← MAIN FILE")
print("=" * 40)
print(f"\nmarketing_master.csv")
print(f"Rows    : {df_master.shape[0]:,}")
print(f"Columns : {df_master.shape[1]}")

✅ All files saved successfully!
📁 Files created:
   1. campaigns_cleaned.csv
   2. performance_cleaned.csv
   3. influencer_cleaned.csv
   4. marketing_master.csv  ← MAIN FILE

marketing_master.csv
Rows    : 242,487
Columns : 42
